In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import warnings
warnings.filterwarnings("ignore")


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

models = {

"Logistic Regression": Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(C=100,class_weight='balanced',max_iter=2000))
]),

"KNN": Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier(metric='euclidean',n_neighbors=3,weights='distance'))
]),

"SVM": Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(C=10,degree=2,gamma=1))
]),

"Random Forest": RandomForestClassifier(
    class_weight='balanced',
    max_features='log2',
    min_samples_split=10,
    n_estimators=300,
    random_state=42
),

"Gradient Boosting": GradientBoostingClassifier(
    max_depth=5,
    n_estimators=200,
    random_state=42
),

"XGBoost": XGBClassifier(
    colsample_bytree=0.7,
    learning_rate=0.05,
    max_depth=7,
    n_estimators=400,
    eval_metric='mlogloss',
    num_class=3
)
}


In [12]:
df=pd.read_excel('WQD.xlsx')
df.head()

,Temp,Turbidity (cm),DO(mg/L),BOD (mg/L),CO2,pH`,Alkalinity (mg L-1 ),Hardness (mg L-1 ),Calcium (mg L-1 ),Ammonia (mg L-1 ),Nitrite (mg L-1 ),Phosphorus (mg L-1 ),H2S (mg L-1 ),Plankton (No. L-1),Water Quality
0,67.448725,10.127148,0.208153,7.473607,10.181084,4.751657,218.364855,300.125080,337.178226,0.286054,4.355310,0.005984,0.066793,6069.624017,2
1,64.626666,94.015595,11.434463,10.859998,14.860521,3.085154,273.939692,8.426776,363.660740,0.096040,2.182753,0.004906,0.023428,250.995959,2
2,65.121842,90.653462,12.430865,12.809970,12.319980,9.648515,220.812730,11.726274,309.370934,0.974501,4.901760,0.006979,0.065041,7218.927473,2
3,1.640334,0.066344,10.963529,8.508023,12.955209,4.819988,266.571628,6.627655,8.180468,0.884865,3.571842,3.174473,0.026018,1230.062252,2
4,64.863434,2.119173,1.361736,13.335372,13.603197,10.244034,252.108000,339.891514,253.996871,0.801695,4.655898,3.854701,0.060995,1035.054820,2


In [13]:
# Clean column names
df.columns = (
    df.columns
    .str.strip()
    .str.replace('`', '', regex=False)
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
    .str.replace('/', '_', regex=False)
    .str.replace('-', '_', regex=False)
    .str.replace(' ', '_')
)

df.columns


Index(['Temp', 'Turbidity_cm', 'DOmg_L', 'BOD_mg_L', 'CO2', 'pH',
       'Alkalinity_mg_L_1_', 'Hardness_mg_L_1_', 'Calcium_mg_L_1_',
       'Ammonia_mg_L_1_', 'Nitrite_mg_L_1_', 'Phosphorus_mg_L_1_',
       'H2S_mg_L_1_', 'Plankton_No._L_1', 'Water_Quality'],
      dtype='object')

In [8]:
X = df.drop(columns=['Water_Quality'])
y = df['Water_Quality']


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
num_classes = y.nunique()

In [14]:
from sklearn.metrics import classification_report, accuracy_score

class_names = ['Excellent','Good','Poor']

rows = []

for name, model in models.items():

    model.fit(X_train, y_train)

    # ---------- TRAINING ----------
    y_train_pred = model.predict(X_train)
    report_train = classification_report(
        y_train, y_train_pred,
        target_names=class_names,
        output_dict=True
    )

    train_acc = accuracy_score(y_train, y_train_pred)

    for c in class_names:
        rows.append({
            "Model": name,
            "Data": "Training",
            "Class": c,
            "Precision": report_train[c]['precision'],
            "Recall": report_train[c]['recall'],
            "F1-score": report_train[c]['f1-score'],
            "Accuracy": train_acc
        })

    # ---------- TESTING ----------
    y_test_pred = model.predict(X_test)
    report_test = classification_report(
        y_test, y_test_pred,
        target_names=class_names,
        output_dict=True
    )

    test_acc = accuracy_score(y_test, y_test_pred)

    for c in class_names:
        rows.append({
            "Model": name,
            "Data": "Testing",
            "Class": c,
            "Precision": report_test[c]['precision'],
            "Recall": report_test[c]['recall'],
            "F1-score": report_test[c]['f1-score'],
            "Accuracy": test_acc
        })

results_table = pd.DataFrame(rows)
results_table = results_table.round(3)

results_table

,Model,Data,Class,Precision,Recall,F1-score,Accuracy
0,Logistic Regression,Training,Excellent,0.883,0.990,0.934,0.839
1,Logistic Regression,Training,Good,0.775,0.910,0.837,0.839
2,Logistic Regression,Training,Poor,0.871,0.631,0.732,0.839
3,Logistic Regression,Testing,Excellent,0.848,0.996,0.916,0.820
4,Logistic Regression,Testing,Good,0.778,0.879,0.826,0.820
5,Logistic Regression,Testing,Poor,0.837,0.600,0.699,0.820
6,KNN,Training,Excellent,1.000,1.000,1.000,1.000
7,KNN,Training,Good,1.000,1.000,1.000,1.000
8,KNN,Training,Poor,1.000,1.000,1.000,1.000
9,KNN,Testing,Excellent,0.903,1.000,0.949,0.869


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

def build_ann(input_dim,
              n_hidden_layers=3,
              units=256,
              dropout_rate=0.4,
              l2_reg=0.0,
              learning_rate=0.0005):

    model = Sequential()

    model.add(Dense(units,
                    activation='relu',
                    kernel_regularizer=l2(l2_reg),
                    input_shape=(input_dim,)))

    model.add(BatchNormalization())
    model.add(Dropout(dropout_rate))

    for _ in range(n_hidden_layers - 1):
        model.add(Dense(units,
                        activation='relu',
                        kernel_regularizer=l2(l2_reg)))
        model.add(BatchNormalization())
        model.add(Dropout(dropout_rate))

    model.add(Dense(num_classes, activation='softmax'))

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [25]:
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler

# Scale data
scaler_ann = StandardScaler()
X_train_dl = scaler_ann.fit_transform(X_train)
X_test_dl = scaler_ann.transform(X_test)

# One-hot encode
y_train_cat = to_categorical(y_train, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

# Build ANN using your best parameters
ann_model = build_ann(
    input_dim=X_train_dl.shape[1],
    n_hidden_layers=3,
    units=256,
    dropout_rate=0.4,
    l2_reg=0.0,
    learning_rate=0.0005
)

In [26]:
history = ann_model.fit(
    X_train_dl,
    y_train_cat,
    validation_split=0.2,   # 20% of train for validation
    epochs=300,
    batch_size=32,
    callbacks=[EarlyStopping(patience=20, restore_best_weights=True)],
    verbose=0
)

In [30]:
# 1️⃣ Generate ANN predictions
y_train_pred_ann = np.argmax(ann_model.predict(X_train_dl), axis=1)
y_test_pred_ann = np.argmax(ann_model.predict(X_test_dl), axis=1)

# 2️⃣ Compute ANN metrics and append to rows
# ---------- ANN TRAIN ----------
report_train_ann = classification_report(
    y_train, y_train_pred_ann,
    target_names=class_names,
    output_dict=True
)
train_acc_ann = accuracy_score(y_train, y_train_pred_ann)

for c in class_names:
    rows.append({
        "Model": "ANN",
        "Data": "Training",
        "Class": c,
        "Precision": report_train_ann[c]['precision'],
        "Recall": report_train_ann[c]['recall'],
        "F1-score": report_train_ann[c]['f1-score'],
        "Accuracy": train_acc_ann
    })

# ---------- ANN TEST ----------
report_test_ann = classification_report(
    y_test, y_test_pred_ann,
    target_names=class_names,
    output_dict=True
)
test_acc_ann = accuracy_score(y_test, y_test_pred_ann)

for c in class_names:
    rows.append({
        "Model": "ANN",
        "Data": "Testing",
        "Class": c,
        "Precision": report_test_ann[c]['precision'],
        "Recall": report_test_ann[c]['recall'],
        "F1-score": report_test_ann[c]['f1-score'],
        "Accuracy": test_acc_ann
    })

# 3️⃣ Create final table
results_table = pd.DataFrame(rows).round(3)
results_table

108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


,Model,Data,Class,Precision,Recall,F1-score,Accuracy
0,Logistic Regression,Training,Excellent,0.883,0.990,0.934,0.839
1,Logistic Regression,Training,Good,0.775,0.910,0.837,0.839
2,Logistic Regression,Training,Poor,0.871,0.631,0.732,0.839
3,Logistic Regression,Testing,Excellent,0.848,0.996,0.916,0.820
4,Logistic Regression,Testing,Good,0.778,0.879,0.826,0.820
5,Logistic Regression,Testing,Poor,0.837,0.600,0.699,0.820
6,KNN,Training,Excellent,1.000,1.000,1.000,1.000
7,KNN,Training,Good,1.000,1.000,1.000,1.000
8,KNN,Training,Poor,1.000,1.000,1.000,1.000
9,KNN,Testing,Excellent,0.903,1.000,0.949,0.869
